In [0]:
from pyspark.sql.functions import current_date

# Read raw CSV from S3
df_raw = (spark.read
          .format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load("s3://wild-project/wildlife_raw_detections_2025_full_year.csv"))

# Add load_date
df_staging = df_raw.withColumn("load_date", current_date())

# Write to staging table (overwrite per batch)
df_staging.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("wild_pro.wildlife_staging.new_wildlife_data")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS wild_pro.bronze.wildlife_detections_raw
USING DELTA
AS SELECT * 
FROM wild_pro.wildlife_staging.new_wildlife_data
WHERE 1=0;

In [0]:
%sql
MERGE INTO wild_pro.bronze.wildlife_detections_raw b
USING wild_pro.wildlife_staging.new_wildlife_data s
ON b.image_id = s.image_id
WHEN NOT MATCHED THEN INSERT *

In [0]:
from pyspark.sql.functions import col, try_to_timestamp, to_date, month, year, initcap

df_staging = spark.table("wild_pro.wildlife_staging.new_wildlife_data")

df_silver_new = (
    df_staging
    .filter(col("confidence").isNotNull())
    .filter(col("confidence") >= 0.6)
    .withColumn("event_time", try_to_timestamp("timestamp"))
    .filter(col("event_time").isNotNull())   # remove invalid timestamps
    .withColumn("species", initcap("species"))
    .withColumn("activity_date", to_date("event_time"))
    .withColumn("month", month("event_time"))
    .withColumn("year", year("event_time"))
)

df_silver_new.createOrReplaceTempView("silver_new_data")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS wild_pro.silver.wildlife_detections_clean
(
    image_id STRING,
    species STRING,
    confidence DOUBLE,
    latitude DOUBLE,
    longitude DOUBLE,
    timestamp STRING,
    camera_id STRING,
    zone STRING,
    load_date DATE,
    event_time TIMESTAMP,
    activity_date DATE,
    month INT,
    year INT
)
USING DELTA
PARTITIONED BY (year, month);

In [0]:
%sql
    
MERGE INTO wild_pro.silver.wildlife_detections_clean s
USING silver_new_data n
ON s.image_id = n.image_id
WHEN NOT MATCHED THEN INSERT *


In [0]:
%sql
CREATE TABLE IF NOT EXISTS wild_pro.gold.fact_wildlife_activity
(
    year INT,
    month INT,
    zone STRING,
    species STRING,
    detection_count BIGINT
)
USING DELTA
PARTITIONED BY (year, month);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW impacted_months AS
SELECT DISTINCT
       year(try_to_timestamp(timestamp)) AS year,
       month(try_to_timestamp(timestamp)) AS month
FROM wild_pro.wildlife_staging.new_wildlife_data
WHERE try_to_timestamp(timestamp) IS NOT NULL
  AND confidence >= 0.6;

In [0]:
%sql
DELETE FROM wild_pro.gold.fact_wildlife_activity g
WHERE EXISTS (
    SELECT 1
    FROM impacted_months i
    WHERE g.year = i.year
      AND g.month = i.month
);

In [0]:
%sql
INSERT INTO wild_pro.gold.fact_wildlife_activity
SELECT
    s.year,
    s.month,
    s.zone,
    s.species,
    COUNT(*) AS detection_count
FROM wild_pro.silver.wildlife_detections_clean s
JOIN impacted_months i
  ON s.year = i.year AND s.month = i.month
GROUP BY
    s.year,
    s.month,
    s.zone,
    s.species;

In [0]:
%sql
OPTIMIZE wild_pro.gold.fact_wildlife_activity
ZORDER BY (zone, species);

In [0]:
%sql
TRUNCATE TABLE wild_pro.wildlife_staging.new_wildlife_data;

In [0]:
%sql
SELECT
    year,
    month,
    SUM(detection_count) AS total_activity
FROM wild_pro.gold.fact_wildlife_activity
GROUP BY year, month
ORDER BY total_activity DESC;

In [0]:
%sql
SELECT
    zone,
    SUM(detection_count) AS total_activity
FROM wild_pro.gold.fact_wildlife_activity
GROUP BY zone
ORDER BY total_activity DESC;

In [0]:
%sql
SELECT
    month,
    species,
    SUM(detection_count) AS species_activity
FROM wild_pro.gold.fact_wildlife_activity
GROUP BY month, species
ORDER BY month;


In [0]:
%sql
OPTIMIZE wild_pro.silver.wildlife_detections_clean
ZORDER BY (zone);

In [0]:
%sql
ALTER TABLE wild_pro.gold.fact_wildlife_activity
SET TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);